<a href="https://colab.research.google.com/github/obeabi/ProjectPortfolio/blob/master/Weekly_Entry_etf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install --upgrade yfinance
!pip install  --upgrade pandas_ta
!pip install ta pandas_ta

In [13]:
import yfinance as yf
print(yf.__version__)
import pandas as pd
#import pandas_ta as ta
import numpy as np
from datetime import datetime
import time
import ta
print("Libraries Installed!")

0.2.64
Libraries Installed!


In [14]:
# List of ETFs to analyze
#df_o = pd.read_csv('stock_list.csv')
df_o = pd.read_csv('etf_list.csv')
#df_o = df_o.sample(20).copy()
#df_o = df_o.head(20).copy()
etfs = df_o['Asset'].to_list()
#etfs =['FEZ', 'VGK', 'EUFN','EWH','ICOP', 'EZA', 'EWN', 'EWS', 'EWI', 'VOO']
print(etfs)

print(len(etfs))

['ARKW', 'ARKK', 'ARKF', 'DAPP', 'EWY', 'IBLC', 'NLR', 'CHAT', 'QBIG', 'BPAY', 'SMH', 'SOXQ', 'SOXX', 'SMHX', 'MNRS', 'ARTY', 'EIS', 'BAI', 'NERD', 'TEK', 'IYW', 'IXN', 'IGM', 'XLK', 'ISRA', 'ESPO', 'EWT', 'PPA', 'VGT', 'IAI', 'BLCR', 'GTEK', 'CRAK', 'PWB', 'QTOP', 'QGRW', 'GMET', 'VOOG', 'ONLN', 'IVW', 'SPYG', 'PNQI', 'ILCG', 'SLV', 'MGK', 'PGRO', 'IBOT', 'RPG', 'VUG', 'AIA', 'IUSG', 'VOX', 'IWF', 'IXP', 'EPOL', 'VONG', 'QQQ', 'QQQM', 'KNCT', 'IWY', 'EWH', 'PPEM', 'EMXC', 'TOPT', 'PSCT', 'EEMA', 'VNQI', 'GGUS', 'XLC', 'IFGL', 'XLG', 'AAXJ', 'BELT', 'HCMT', 'TAN', 'OEF', 'EEMX', 'GBUY', 'DGIN', 'BJK', 'EPU', 'MGC', 'EWP', 'ICOP', 'SLX', 'IOO', 'SPXV', 'EWN', 'JBL', 'STX', 'ORCL', 'AMD', 'WDC', 'MU', 'GEV', 'AVGO', 'MCHP', 'SMCI', 'NVDA', 'AXON', 'PWR', 'ON', 'ANET', 'LRCX', 'APH', 'NFLX', 'HWM', 'KLAC', 'CRWD', 'AMAT', 'META', 'DELL', 'ETN', 'GNRC', 'MPWR', 'MSFT', 'EMR', 'DIS', 'NTAP', 'SNPS', 'IBM', 'TXN', 'GE', 'CAT', 'JCI', 'WBD', 'NXPI', 'ADI', 'CAH', 'LYV', 'BA', 'TER', 'ROK', 'S

In [15]:
# inspect dataframe
#df_o.head()

In [16]:
# Function to fetch historical weekly data

def get_monthly_data(ticker):
  try:
      df = yf.download(ticker, period="10y", interval="1mo",auto_adjust=True)
      df['10_month_SMA'] = df['Close'].rolling(window=10).mean()
      #df['above_10_month_SMA'] = df['Close'] > df['10_month_SMA']  # Convert to boolean explicitly
      # Calculate monthly ROC (based on 3 trading months)
      df['ROC_1M'] = (df['Close'].pct_change(periods=4)) *100
      # Check if ROC is positive
      df["ROC_1M_Positive"] = df["ROC_1M"] > 0
      df['ROC_1M_SMA'] = df['ROC_1M'].rolling(window=3).mean()
      # We take the difference between ROC of the current month and the ROC of the past N months
      df["ROC_1M_Slope_Positive"] = np.where(df['ROC_1M_SMA'] > df['ROC_1M_SMA'].shift(1), True, False)

      # --- Overhead Resistance Filter ---
      recent_12_months = df[-12:]
      max_close_12m = recent_12_months['Close'].max().iloc[0]
      last_close = df['Close'].iloc[-1].iloc[0]
      # Filter condition: Close is within 10% of 12-month high
      df['No_Overhead_Resistance'] = last_close >= (max_close_12m * 0.9)

      # --- Untraded zone Filter ---
      recent_120_months = df[-120:]
      max_close_120m = recent_120_months['Close'].max().iloc[0]
      # Filter condition: Close is within 10% of 12-month high
      df['No_10y_Overhead_Resistance'] = last_close >= max_close_120m

      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      #df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
      #df["ROC_1M"].rolling(window=8).apply(
        #lambda x: (x[-1] > x[0]), raw=True).astype(bool)  # Convert to boolean explicitly
      return df
  except Exception as e:
      print("There is an error getting monthly data", e)

def get_weekly_data(ticker):
  try:
      df = yf.download(ticker, period="2y", interval="1wk",auto_adjust=True)
      df['20_week_SMA'] = df['Close'].rolling(window=20).mean()
      df['50_week_SMA'] = df['Close'].rolling(window=50).mean()
      df['RSI'] = compute_rsi(df['Close'])
      df['ATR'] = compute_atr(df, 14)
      df['OBV'] = compute_obv(df)
      df['10_week_avg_volume'] = df['Volume'].rolling(window=10).mean()
      df['ma'] = calculate_ma(df['RSI'])
      df['upper_band'], df['lower_band'] = calculate_bollinger_bands(df['RSI'])
      # Calculate MACD and Signal Line
      df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
      df['21_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
      df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
      # Compute MACD Line
      df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
      # Compute Signal Line (9-day EMA of MACD Line)
      df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
      df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
      # Compute raw EFI
      df['EFI'] = (df['Close'].diff()) * df['Volume']
      # Compute EMA of EFI
      df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
      # Determine if EFI_EMA is rising or falling
      df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')

      # Calculate weekly ROC (based on 9 trading  week)
      df['ROC_1W'] = (df['Close'].pct_change(periods=9)) *100
      # Check if ROC is positive
      df["ROC_1W_Positive"] = df["ROC_1W"] > 0
      df['ROC_1W_SMA'] = df['ROC_1W'].rolling(window=3).mean()
      # We take the difference between ROC of the current week and the ROC of the past N weeks
      df["ROC_1W_Slope_Positive"] = np.where(df['ROC_1W_SMA'] > df['ROC_1W_SMA'].shift(1), True, False)
      return df
  except Exception as e:
      print("There is an error getting weekly data", e)

def is_macd_bullish(df):
    """
    Determines if there is a bullish signal on the MACD indicator.
    A bullish signal occurs when the MACD line is above the signal line.

    Parameters:
    - df: dataframe. Must have columns 'MACD_Line' and 'Signal_Line'.
    Returns:
    - bool: True if a bullish signal is detected, otherwise False.
    """
    try:
      if df.empty:
        return False
      # Calculate MACD histogram if not already present

      curr = df.iloc[-3:]
      macd_crossover = curr['MACD_Line'].iloc[-1] > curr['Signal_Line'].iloc[-1]
      hist_increasing = (curr['MACD_Hist'].iloc[-1] > curr['MACD_Hist'].iloc[-2]) \
                         or (curr['MACD_Hist'].iloc[-2] > curr['MACD_Hist'].iloc[-3])

      return macd_crossover and hist_increasing
    except Exception as e:
      print("Something went wrong while computing the MACD", e)


def calculate_bbw(df, window=20):
    """ Calculates Bollinger Bands Width"""
    sma = df['Close'].rolling(window).mean()
    std = df['Close'].rolling(window).std()
    upper = sma + 2*std
    lower = sma - 2*std
    bbw = (upper - lower) / sma
    return bbw

def calculate_vwap(df):
  """ Calculates vwap"""
  try:
    Typical_Price = (df['Close'].values + df['High'].values + df['Low'].values) / 3
    TPV = Typical_Price * df['Volume'].values
    vwap = TPV.cumsum() / df['Volume'].values.cumsum()
    return vwap
  except Exception as e:
    print("Something went wrong while computing the VWAP:", e)
    return None

# Function to compute RSI
def compute_rsi(series, period=10):
  try:
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))
  except Exception as e:
        print("Something went wrong while computing the RSI:", e)
        return None

def calculate_ma(data, length=10, ma_type="WMA"):
    if ma_type == "SMA":
        return data.rolling(window=length).mean()
    elif ma_type == "EMA":
        return data.ewm(span=length, adjust=False).mean()
    elif ma_type == "WMA":
        weights = np.arange(1, length+1)
        return data.rolling(length).apply(lambda x: np.dot(x, weights)/weights.sum(), raw=True)
    elif ma_type == "VWMA":
        return ta.volume_weighted_average_price(data, length)

def calculate_bollinger_bands(data, length=10, std_dev=2.0):
    sma = data.rolling(window=length).mean()
    std_dev = data.rolling(window=length).std()
    upper_band = sma + (std_dev * std_dev)
    lower_band = sma - (std_dev * std_dev)
    return upper_band, lower_band

# Function to compute ATR (Average True Range)
def compute_atr(df, period=10):
  try:
    df['High-Low'] = df['High'] - df['Low']
    df['High-Close'] = abs(df['High'] - df['Close'].shift(1))
    df['Low-Close'] = abs(df['Low'] - df['Close'].shift(1))
    df['TR'] = df[['High-Low', 'High-Close', 'Low-Close']].max(axis=1)
    return df['TR'].rolling(window=period).mean()
  except Exception as e:
      print("Something went wrong whilecomputing the ATR", e)


# Function to compute On-Balance Volume (OBV)
def compute_obv(df):
  try:
    # Calculate daily price change: 1 if price is up, -1 if down, 0 if unchanged
    price_change = df['Close'].diff()

    # Use price change to decide whether to add or subtract volume
    obv = (price_change > 0).astype(int) * df['Volume']  # Volume when price goes up
    obv -= (price_change < 0).astype(int) * df['Volume']  # Volume when price goes down

    # We accumulate the OBV by taking the cumulative sum of the volume changes
    obv = obv.cumsum()

    return obv
  except Exception as e:
      print("Something went wrong while computing the OBV", e)

def is_bullish_engulfing(df):
    prev = df.iloc[-2]
    curr = df.iloc[-1]
    return (
        prev['Close'].iloc[0] < prev['Open'].iloc[0] and # Previous red
        curr['Close'].iloc[0] > curr['Open'].iloc[0] and # Current green
        curr['Close'].iloc[0] > prev['Open'].iloc[0] and
        curr['Open'].iloc[0] < prev['Close'].iloc[0]
    )

# Function to calculate risk-reward ratio
def calculate_risk_reward(df):
  try:
    if df.empty or len(df) < 20:  # Ensure there are enough data points
        return np.nan

    latest_price = df['Close'].iloc[-1].iloc[0]

    # Use the ATR for setting support level
    atr = df['ATR'].iloc[-1]  # Latest ATR value
    price_ema = df['8_day_EMA'].iloc[-1]
    atr_multiple = 1.5  # You can adjust this multiplier based on your strategy

    # Calculate the support level using the ATR
    trailing = atr * atr_multiple
    support_level = price_ema - trailing
    risk = np.abs(latest_price- support_level)
    resistance_level = latest_price + (2*risk)
    reward = resistance_level - latest_price

    # Ensure risk is greater than zero before division
    if risk > 0:
        risk_reward_ratio = reward / risk
        return risk_reward_ratio if risk_reward_ratio > 0 else np.nan , support_level, resistance_level, latest_price, trailing
    else:
        return np.nan,np.nan, np.nan, np.nan, np.nan
  except Exception as e:
      print("Something went wrong while computing the reward-risk ratio", e)

# Function to fetch daily data
def get_daily_data(ticker):
    df = yf.download(ticker, period="1y", interval="1d",auto_adjust=True)
    df['20_day_SMA'] = df['Close'].rolling(window=20).mean()
    df['20_day_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['5_day_EMA'] = df['Close'].ewm(span=5, adjust=False).mean()
    df['8_day_EMA'] = df['Close'].ewm(span=8, adjust=False).mean()
    df['13_day_EMA'] = df['Close'].ewm(span=13, adjust=False).mean()
    df['21_day_EMA'] = df['Close'].ewm(span=21, adjust=False).mean()
    df['26_day_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    df['50_day_SMA'] = df['Close'].rolling(window=50).mean()
    df["Distance_EMA"] = (df['Close']/ df['Close'].ewm(span=20, adjust=False).mean() ) - 1
    df['RSI'] = compute_rsi(df['Close'],period=10)
    df['ATR'] = compute_atr(df, 10)
    df["EMA_plus_ATR"] = df["8_day_EMA"] + 1.0* df["ATR"]
    df['ma'] = calculate_ma(df['RSI'])
    df['BBW'] = calculate_bbw(df)
    df['upper_band'], df['lower_band'] = calculate_bollinger_bands(df['RSI'])
    # Calculate MACD and Signal Line
    df['12_EMA'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['26_EMA'] = df['Close'].ewm(span=26, adjust=False).mean()
    # Compute MACD Line
    df['MACD_Line'] = df['12_EMA'] - df['26_EMA']
    # Compute Signal Line (9-day EMA of MACD Line)
    df['Signal_Line'] = df['MACD_Line'].ewm(span=9, adjust=False).mean()
    df['MACD_Hist'] = df['MACD_Line'] - df['Signal_Line']
    df['VWAP'] = calculate_vwap(df)
    # Compute raw EFI
    df['EFI'] = (df['Close'].diff()) * df['Volume']
    # Compute EMA of EFI
    df['EFI_EMA'] = df['EFI'].ewm(span=13, adjust=False).mean()
    # Determine if EFI_EMA is rising or falling
    df['EFI_EMA_Trend'] = df['EFI_EMA'].diff().apply(lambda x: 'Rising' if x > 0 else 'Falling')
    # Calculate daily ROC (based on 10 trading days per week)
    df['ROC'] = (df['Close'].pct_change(periods=10)) *100
    # Check if ROC is positive
    df["ROC_Positive"] = df["ROC"] > 0
    df['ROC_SMA'] = df['ROC'].rolling(window=3).mean()
    # We take the difference between ROC of the current day and the ROC of the past N days
    df["ROC_Slope_Positive"] = np.where(df['ROC_SMA'] > df['ROC_SMA'].shift(1), True, False)
    return df

# Function to fetch hourly data
def get_1hr_data(ticker):
    df = yf.download(ticker, interval='1h', period='60d',auto_adjust=True)
    df['20_hr_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['50_hr_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    df['100_hr_EMA'] = df['Close'].ewm(span=100, adjust=False).mean()
    df['200_hr_EMA'] = df['Close'].ewm(span=200, adjust=False).mean()
    return df

# Function to fetch 15mins data to refine entry
def get_15min_data(ticker):
    df = yf.download(ticker, interval='15m', period='60d',auto_adjust=True)
    df['20_min_EMA'] = df['Close'].ewm(span=20, adjust=False).mean()
    df['50_min_EMA'] = df['Close'].ewm(span=50, adjust=False).mean()
    return df

# Function to check monthly trend
def is_monthly_trend_bullish(df):
    if df.empty:
        return False

    overhead_flag = df['No_Overhead_Resistance'].iloc[-1]
    new_range_flag = df['No_10y_Overhead_Resistance'].iloc[-1]
    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['10_month_SMA'].iloc[-1]
    roc_above_zero = df['ROC_1M_Positive'].iloc[-1]
    roc_trend_ok = df['ROC_1M_Slope_Positive'].iloc[-1]
    macd_bullish_signal = is_macd_bullish(df)
    above_10_month_SMA = latest_price > latest_sma
    return above_10_month_SMA and roc_above_zero and roc_trend_ok and overhead_flag \
               and macd_bullish_signal or new_range_flag

# Function to check weekly trend
def is_weekly_trend_bullish(df):
    if df.empty:
        return False

    latest_price = df['Close'].iloc[-1].iloc[0]
    latest_sma = df['20_week_SMA'].iloc[-1]
    above_20_week_SMA = latest_price > latest_sma
    #latest_rsi = df['RSI'].iloc[-1]
    macd_bullish_signal = is_macd_bullish(df)
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Rising'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] > 0
    rocw_above_zero = df['ROC_1W_Positive'].iloc[-1]
    rocw_trend_ok = df['ROC_1W_Slope_Positive'].iloc[-1]
    volume_ok = df['Volume'].iloc[-1] > df['10_week_avg_volume'].iloc[-1] # Institutional interest
    volume_ok = volume_ok.iloc[0]
    # Calculate the OBV Moving Average
    df['OBV_EMA'] = df['OBV'].ewm(span=10, adjust=False).mean()
    # OBV trending up if current OBV is above the 20-period EMA
    obv_trending_up = df['OBV'].iloc[-1] > df['OBV_EMA'].iloc[-1]
    # obv_trending_up = df['OBV'].iloc[-1] > df['OBV'].iloc[-5] # OBV increasing over last 5 weeks

    # OBV trending down if current OBV is below the 20-period EMA
    obv_trending_down = df['OBV'].iloc[-1] < df['OBV_EMA'].iloc[-1]
    trend_ok =above_20_week_SMA
    rsi_ok = df['RSI'].iloc[-1] >= 50 # Not  oversold
    time.sleep(2)  # Add a delay of 1 second between requests

    return  trend_ok and rsi_ok and rocw_above_zero and (volume_ok or obv_trending_up) \
            and elderforce_trend_ok and elderforce_ema_ok and macd_bullish_signal

# Function to check daily entry signal
def is_daily_entry_signal(df2):
    if df2.empty:
        return False

    # Apply BBW filter (must be > median)
    median_bbw = df2['BBW'].rolling(20).median()
    df2['BBW_Filter'] = df2['BBW'] > median_bbw
    # Filter to generate signal
    df = df2[df2['BBW_Filter']].copy()
    latest_price = df['Close'].iloc[-1].iloc[0]
    prev_price = df['Close'].iloc[-2].iloc[0]
    latest_sma = df['20_day_SMA'].iloc[-1]
    latest_50sma = df['50_day_SMA'].iloc[-1]
    above_50SMA = latest_price > latest_50sma
    latest_rsi = df['RSI'].iloc[-1]
    latest_distance_20ema = df['Distance_EMA'].iloc[-1]
    macd_bullish_signal = is_macd_bullish(df)
    vwap_price = df['VWAP'].iloc[-1]
    elderforce_trend_ok = df['EFI_EMA_Trend'].iloc[-1] == 'Rising'
    elderforce_ema_ok = df['EFI_EMA'].iloc[-1] > 0
    rocd_above_zero = df['ROC_Positive'].iloc[-1]
    rocd_trend_ok = df['ROC_Slope_Positive'].iloc[-1]


    # Look for a breakout above 20-day SMA & RSI > 50
    return above_50SMA and (latest_rsi > 50) and macd_bullish_signal \
            and elderforce_trend_ok and elderforce_ema_ok or (latest_price > vwap_price)

# Check entry conditions
def check_entry_conditions(tickers):
    results = []
    for ticker in tickers:
      df = get_daily_data(ticker)
      latest_price = df['Close'].iloc[-1].iloc[0]
      prev_price = df['Close'].iloc[-2].iloc[0]
      latest_sma = df['50_day_SMA'].iloc[-1]
      latest_rsi = df['RSI'].iloc[-1]
      latest_distance_20ema = df['Distance_EMA'].iloc[-1]
      latest_price_5ema =df['5_day_EMA'].iloc[-1]
      latest_price_8ema =df['8_day_EMA'].iloc[-1]
      latest_price_13ema =df['13_day_EMA'].iloc[-1]
      latest_price_26ema =df['26_day_EMA'].iloc[-1]
      price_threshold_ATR = df['EMA_plus_ATR'].iloc[-1]
      above_price_threshold_ATR = latest_price > price_threshold_ATR
      below_price_threshold_ATR = latest_price <= price_threshold_ATR
      df_entry = get_1hr_data(ticker)
      df_refine_entry = get_15min_data(ticker)
      latest_priceh_20ema = df_entry['20_hr_EMA'].iloc[-1]
      latest_priceh_50ema = df_entry['50_hr_EMA'].iloc[-1]
      latest_priceh_100ema = df_entry['100_hr_EMA'].iloc[-1]
      latest_priceh_200ema = df_entry['200_hr_EMA'].iloc[-1]
      latest_priceh        = df_refine_entry['Close'].iloc[-1].iloc[0]

      latest_pricem_20ema =df_refine_entry['20_min_EMA'].iloc[-1]
      latest_pricem_50ema =df_refine_entry['50_min_EMA'].iloc[-1]
      latest_pricem       = df_refine_entry['Close'].iloc[-1].iloc[0]



      refined_entry_signal = (latest_priceh >  latest_priceh_50ema) and \
                             (latest_priceh_50ema > latest_priceh_100ema) and \
                             (latest_pricem >  latest_pricem_50ema) and \
                             (latest_pricem_20ema > latest_pricem_50ema)



      if latest_price >= latest_price_8ema and above_price_threshold_ATR and refined_entry_signal:
        entry_signal = "Extended Momentum Entry"
      elif latest_price >= latest_price_8ema and below_price_threshold_ATR and refined_entry_signal :
        entry_signal = "Aline Entry"
      elif (latest_price < latest_price_8ema) and (latest_price >= latest_price_13ema) and refined_entry_signal:
        entry_signal = "Pullback Entry"
      elif (latest_price <= latest_price_13ema) and (latest_price >= latest_price_26ema) and refined_entry_signal :
          entry_signal= "Elder Entry"
      elif  (latest_price <= latest_price_26ema) and (latest_price >= latest_sma) and refined_entry_signal:
          entry_signal = "Below Elder Entry"
      else:
        entry_signal = "Bearish"
      results.append([ticker, entry_signal])
    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results

# Multi-timeframe strategy check returning a DataFrame
def check_mtf_entry(tickers):
    results = []

    for ticker in tickers:
        monthly_df = get_monthly_data(ticker)
        weekly_df = get_weekly_data(ticker)
        daily_df = get_daily_data(ticker)

        if is_monthly_trend_bullish(monthly_df) and is_weekly_trend_bullish(weekly_df):
            if is_daily_entry_signal(daily_df):
                entry_signal = "Entry Confirmed ✅"
            else:
                entry_signal = "No Entry Yet on Daily Timeframe ⏳"
        else:
            entry_signal = "Monthly or Weekly Trend Not Bullish ❌"

        results.append([ticker, entry_signal])
        time.sleep(2)  # Add a delay of 1 second between requests

    # Convert results to DataFrame
    df_results = pd.DataFrame(results, columns=["Asset", "Entry_Signal"])
    return df_results



In [17]:

# Multi-time frame entry Check
#df_results = pd.DataFrame(results).dropna()
etfs_to_check = df_o['Asset'].tolist()  #df_o['ETF'] #df_results['ETF'].tolist()
df_signals = check_mtf_entry(etfs_to_check)

df_signals.head()



[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Asset,Entry_Signal
0,ARKW,Entry Confirmed ✅
1,ARKK,Entry Confirmed ✅
2,ARKF,Entry Confirmed ✅
3,DAPP,No Entry Yet on Daily Timeframe ⏳
4,EWY,Entry Confirmed ✅


## Generate buy list

In [18]:
df_final = df_signals[df_signals['Entry_Signal'] =="Entry Confirmed ✅"]
final_etfs_to_check = df_final['Asset'].tolist()


buy_list = check_entry_conditions(final_etfs_to_check)

#buy_list.head()
buy_list

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Asset,Entry_Signal
0,ARKW,Bearish
1,ARKK,Bearish
2,ARKF,Bearish
3,EWY,Aline Entry
4,IBLC,Extended Momentum Entry
...,...,...
104,CSCO,Aline Entry
105,TT,Bearish
106,EA,Bearish
107,VRSN,Aline Entry


In [19]:
# Apply TA filters and prioritize ETFs
results = []
buy_list = buy_list[buy_list['Entry_Signal'].isin([
    #'Extended Momentum Entry',
    'Aline Entry',
    #'Momentum Entry',
    'Pullback Entry',
    #'Elder Entry'
])]


for etf in buy_list['Asset'].to_list():
   df =get_daily_data(etf)
   price = df['Close'].iloc[-1].iloc[0]
   above_21EMA = price > df['21_day_EMA'].iloc[-1]
   above_50sma = price  > df['50_day_SMA'].iloc[-1]
   price_21ema = df['21_day_EMA'].iloc[-1]
   price_13ema = df['13_day_EMA'].iloc[-1]



   if  above_50sma  :
    rr_ratio,support_level, resistance_level, latest_price,trail = calculate_risk_reward(df)

    entry_price = latest_price +(0.2*trail )
    new_resistance_level = resistance_level+(0.2*trail )
    new_support_level = support_level+(0.2*trail )
    stop_loss_perc = ((new_support_level- entry_price)/entry_price )*100
    take_profit_perc = ((new_resistance_level- entry_price )/entry_price )*100
    # Fetch the Entry_Signal from buy_list
    entry_signal = buy_list.loc[buy_list['Asset'] == etf, 'Entry_Signal'].values[0]
    # Append results with Entry_Signal
    results.append({
            "Asset": etf,
            "Risk-Reward": rr_ratio,
            "Stop Loss": new_support_level,
            "Take Profit": new_resistance_level,
            "Current Price": latest_price,
            "Entry Price": entry_price,
            "Trail Price": trail,
            "Entry Signal": entry_signal,  # Add entry signal
            "stop_loss_perc": stop_loss_perc,
            "take_profit_perc": take_profit_perc
        })

    time.sleep(2)  # Add a delay of 1 second between requests


# Sort ETFs by highest risk-to-reward ratio
try:
   df_results = pd.DataFrame(results).dropna().sort_values(by="Risk-Reward", ascending=False).reset_index(drop = True)
except Exception as e:
  print("No Asset to buy today, check back some other time!")
  df_results = pd.DataFrame({"Asset": ["No Asset available"]})

df2 = df_results.merge(df_o[['Asset','Type', 'score']], on='Asset', how='left')
df2['timestamp'] = datetime.now()
df2 = df2.sort_values(by='score', ascending=False)
df2.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Type,score,timestamp
13,ORCL,2.0,203.003652,257.933217,218.960007,221.313507,11.767502,Aline Entry,-8.273266,16.546532,Stock,2.64,2025-07-02 03:19:23.205267
14,LRCX,2.0,92.487254,107.706732,96.809998,97.560413,3.752079,Aline Entry,-5.200018,10.400037,Stock,1.50,2025-07-02 03:19:23.205267
0,EWY,2.0,69.045589,78.150021,71.639999,72.080400,2.202002,Aline Entry,-4.210313,8.420626,ETF,1.37,2025-07-02 03:19:23.205267
15,KLAC,2.0,857.968645,1002.998345,898.849976,906.311879,37.309515,Aline Entry,-5.334062,10.668123,Stock,1.28,2025-07-02 03:19:23.205267
16,BA,2.0,200.005333,234.248112,209.789993,211.419593,8.147999,Aline Entry,-5.398866,10.797731,Stock,0.52,2025-07-02 03:19:23.205267


 # ETF Entries

In [20]:
# Fetch the Entry_Signal from buy_list
df2[df2['Type'] == 'ETF'].reset_index(drop=True)

,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Type,score,timestamp
0,EWY,2.0,69.045589,78.150021,71.639999,72.080400,2.202002,Aline Entry,-4.210313,8.420626,ETF,1.37,2025-07-02 03:19:23.205267
1,EWT,2.0,56.045834,61.410637,57.580002,57.834101,1.270498,Aline Entry,-3.092064,6.184129,ETF,0.46,2025-07-02 03:19:23.205267
2,ISRA,2.0,49.635560,56.005580,51.500000,51.758900,1.294500,Aline Entry,-4.102367,8.204735,ETF,0.46,2025-07-02 03:19:23.205267
3,BLCR,2.0,35.628189,37.651215,36.220001,36.302531,0.412650,Aline Entry,-1.857562,3.715123,ETF,0.41,2025-07-02 03:19:23.205267
4,GTEK,2.0,34.990898,37.775478,35.790001,35.919091,0.645451,Aline Entry,-2.584123,5.168246,ETF,0.37,2025-07-02 03:19:23.205267
5,PGRO,2.0,39.861139,42.597114,40.651001,40.773131,0.610649,Aline Entry,-2.236747,4.473493,ETF,0.26,2025-07-02 03:19:23.205267
6,AIA,2.0,79.394596,85.829002,81.239998,81.539398,1.496999,Aline Entry,-2.630388,5.260775,ETF,0.23,2025-07-02 03:19:23.205267
7,EWH,2.0,19.503157,20.927084,19.910000,19.977800,0.338999,Aline Entry,-2.375849,4.751698,ETF,0.19,2025-07-02 03:19:23.205267
8,EEMA,2.0,81.054864,86.844075,82.720001,82.984601,1.323000,Aline Entry,-2.325415,4.650831,ETF,0.17,2025-07-02 03:19:23.205267
9,GGUS,2.0,56.282387,59.895961,57.313999,57.486912,0.864562,Aline Entry,-2.095302,4.190604,ETF,0.15,2025-07-02 03:19:23.205267


# Stock Entries

In [21]:
# Fetch the Entry_Signal from buy_list
df2[df2['Type'] == 'Stock'].reset_index(drop=True)


,Asset,Risk-Reward,Stop Loss,Take Profit,Current Price,Entry Price,Trail Price,Entry Signal,stop_loss_perc,take_profit_perc,Type,score,timestamp
0,ORCL,2.0,203.003652,257.933217,218.960007,221.313507,11.767502,Aline Entry,-8.273266,16.546532,Stock,2.64,2025-07-02 03:19:23.205267
1,LRCX,2.0,92.487254,107.706732,96.809998,97.560413,3.752079,Aline Entry,-5.200018,10.400037,Stock,1.50,2025-07-02 03:19:23.205267
2,KLAC,2.0,857.968645,1002.998345,898.849976,906.311879,37.309515,Aline Entry,-5.334062,10.668123,Stock,1.28,2025-07-02 03:19:23.205267
3,BA,2.0,200.005333,234.248112,209.789993,211.419593,8.147999,Aline Entry,-5.398866,10.797731,Stock,0.52,2025-07-02 03:19:23.205267
4,CSCO,2.0,67.039112,74.092072,69.099998,69.390099,1.450501,Aline Entry,-3.388073,6.776145,Stock,0.33,2025-07-02 03:19:23.205267
5,VRSN,2.0,279.981512,311.566337,289.079987,290.509787,7.149001,Aline Entry,-3.624069,7.248138,Stock,0.24,2025-07-02 03:19:23.205267
